# Stage 1B: Tile Province-Level Images to Prediction Points

Takes your existing province-level quarterly Sentinel-2 composites and crops a ~10x10 km tile centered on each prediction point.

**Input:**
- `prediction_points.csv` (from Stage 1)
- Province-level quarterly GeoTIFFs (from Google Drive)

**Output:**
- Per-point tile folders: `{PointID}/point_{PointID}_2025_Q1.tif` etc.

In [ ]:
import os
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import from_bounds
from rasterio.transform import from_bounds as transform_from_bounds
from pyproj import Transformer
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

# Prediction points from Stage 1
POINTS_CSV = '/Users/ruben/Desktop/Thesis/2025-Data/prediction_points.csv'

# Root folder with province-level quarterly composites.
# Expected structure:
#   IMAGE_ROOT/
#     Ilocos_Norte/
#       Ilocos_Norte_2025_Q1.tif
#       Ilocos_Norte_2025_Q2.tif
#       Ilocos_Norte_2025_Q3.tif
#       Ilocos_Norte_2025_Q4.tif
#     Pampanga/
#       ...
#
# Adjust PROVINCE_IMAGE_MAP below if your naming differs.
IMAGE_ROOT = '/Users/ruben/Desktop/Thesis/2025-Data/Sentinel2'

# Output folder for tiles
TILE_OUTPUT = '/Users/ruben/Desktop/Thesis/2025-Data/tiles'
os.makedirs(TILE_OUTPUT, exist_ok=True)

# Tile size: half-width in meters from center point
# 5000m = 10km x 10km tile (matches training data)
TILE_HALF_SIZE_M = 5000

# Target tile pixel size (match VGG16 input after resize)
# We save the raw crop, then Phase 2 resizes to 224x224.
# No need to enforce pixel size here.

QUARTERS = ['Q1', 'Q2', 'Q3', 'Q4']

print("Configuration loaded.")

In [ ]:
# ============================================================
# 2. MAP PROVINCES TO IMAGE FILES
# ============================================================
# Maps the Province name in prediction_points.csv to the folder
# and filename pattern on disk.
# EDIT the folder names and filename patterns to match yours.

PROVINCE_IMAGE_MAP = {
    'Ilocos Norte': {
        'folder': 'Ilocos_Norte',
        'pattern': 'Ilocos_Norte_2025_{q}.tif'
    },
    'Pampanga': {
        'folder': 'Pampanga',
        'pattern': 'Pampanga_2025_{q}.tif'
    },
    'Benguet': {
        'folder': 'Benguet',
        'pattern': 'Benguet_2025_{q}.tif'
    },
    'Kalinga': {
        'folder': 'Kalinga',
        'pattern': 'Kalinga_2025_{q}.tif'
    },
    'Aklan': {
        'folder': 'Aklan',
        'pattern': 'Aklan_2025_{q}.tif'
    },
    'Zamboanga del Norte': {
        'folder': 'Zamboanga_del_Norte',
        'pattern': 'Zamboanga_del_Norte_2025_{q}.tif'
    },
    'Basilan': {
        'folder': 'Basilan',
        'pattern': 'Basilan_2025_{q}.tif'
    },
    'Tawi-Tawi': {
        'folder': 'Tawi-Tawi',
        'pattern': 'Tawi-Tawi_2025_{q}.tif'
    },
    'Maguindanao del Sur': {
        'folder': 'Maguindanao_del_Sur',
        'pattern': 'Maguindanao_del_Sur_2025_{q}.tif'
    },
    'Davao Oriental': {
        'folder': 'Davao_Oriental',
        'pattern': 'Davao_Oriental_2025_{q}.tif'
    },
    'Metropolitan Manila': {
        'folder': 'NCR',
        'pattern': 'NCR_2025_{q}.tif'
    },
}

# Verify all image files exist
print("Checking image files...")
missing = []
for prov, info in PROVINCE_IMAGE_MAP.items():
    for q in QUARTERS:
        fpath = os.path.join(
            IMAGE_ROOT, info['folder'],
            info['pattern'].format(q=q)
        )
        if os.path.exists(fpath):
            print(f"  OK   : {fpath}")
        else:
            print(f"  MISS : {fpath}")
            missing.append(fpath)

if missing:
    print(f"\nWARNING: {len(missing)} files missing. Fix paths before proceeding.")
else:
    print(f"\nAll {len(PROVINCE_IMAGE_MAP) * 4} image files found.")

In [ ]:
# ============================================================
# 3. LOAD PREDICTION POINTS
# ============================================================

df_points = pd.read_csv(POINTS_CSV)
print(f"Loaded {len(df_points)} prediction points.")
print(f"Provinces: {df_points['Province'].nunique()}")
print(df_points['Province'].value_counts())

In [ ]:
# ============================================================
# 4. TILING FUNCTION
# ============================================================

# Transformer: WGS84 lat/lon -> UTM 51N (meters)
to_utm = Transformer.from_crs('EPSG:4326', 'EPSG:32651', always_xy=True)
to_wgs = Transformer.from_crs('EPSG:32651', 'EPSG:4326', always_xy=True)


def crop_tile(src_path, center_lon, center_lat, half_size_m, out_path):
    """
    Crop a tile from a province-level GeoTIFF centered on (lon, lat).
    The bounding box is computed in UTM meters, then converted
    to the source image's CRS for the actual crop.
    """
    # Convert center to UTM
    cx, cy = to_utm.transform(center_lon, center_lat)

    # Bounding box in UTM
    minx = cx - half_size_m
    maxx = cx + half_size_m
    miny = cy - half_size_m
    maxy = cy + half_size_m

    with rasterio.open(src_path) as src:
        src_crs = src.crs

        # If source is not UTM, convert bounds to source CRS
        if str(src_crs) != 'EPSG:32651':
            from pyproj import Transformer as T
            utm_to_src = T.from_crs('EPSG:32651', src_crs, always_xy=True)
            minx_s, miny_s = utm_to_src.transform(minx, miny)
            maxx_s, maxy_s = utm_to_src.transform(maxx, maxy)
        else:
            minx_s, miny_s = minx, miny
            maxx_s, maxy_s = maxx, maxy

        # Clamp to source bounds
        src_bounds = src.bounds
        minx_s = max(minx_s, src_bounds.left)
        miny_s = max(miny_s, src_bounds.bottom)
        maxx_s = min(maxx_s, src_bounds.right)
        maxy_s = min(maxy_s, src_bounds.top)

        # Check if there is any valid overlap
        if minx_s >= maxx_s or miny_s >= maxy_s:
            return False

        # Get the pixel window
        window = from_bounds(
            minx_s, miny_s, maxx_s, maxy_s,
            transform=src.transform
        )

        # Read the data
        # Read only RGB bands (first 3) or all bands
        n_bands = min(src.count, 3)  # Take RGB if available
        data = src.read(
            list(range(1, n_bands + 1)),
            window=window
        )

        if data.size == 0 or data.shape[1] == 0 or data.shape[2] == 0:
            return False

        # Compute output transform
        out_transform = rasterio.transform.from_bounds(
            minx_s, miny_s, maxx_s, maxy_s,
            data.shape[2], data.shape[1]
        )

        # Write tile
        profile = src.profile.copy()
        profile.update({
            'width': data.shape[2],
            'height': data.shape[1],
            'count': n_bands,
            'transform': out_transform,
        })

        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(data)

    return True


print("Tiling function ready.")

In [ ]:
# ============================================================
# 5. RUN TILING
# ============================================================

print("Starting tiling...")
results = []
total = len(df_points) * len(QUARTERS)
done = 0
failed = 0

for _, row in df_points.iterrows():
    pid = int(row['PointID'])
    lat = row['Latitude']
    lon = row['Longitude']
    province = row['Province']

    if province not in PROVINCE_IMAGE_MAP:
        print(f"  SKIP: Province '{province}' not in image map.")
        failed += len(QUARTERS)
        continue

    info = PROVINCE_IMAGE_MAP[province]

    # Create output folder for this point
    pid_str = str(pid).zfill(5)
    out_dir = os.path.join(TILE_OUTPUT, pid_str)
    os.makedirs(out_dir, exist_ok=True)

    for q in QUARTERS:
        src_path = os.path.join(
            IMAGE_ROOT, info['folder'],
            info['pattern'].format(q=q)
        )
        out_path = os.path.join(
            out_dir, f'point_{pid_str}_2025_{q}.tif'
        )

        if os.path.exists(out_path):
            # Skip already processed
            done += 1
            continue

        try:
            success = crop_tile(
                src_path, lon, lat,
                TILE_HALF_SIZE_M, out_path
            )
            if success:
                done += 1
            else:
                failed += 1
                results.append({
                    'PointID': pid, 'Quarter': q,
                    'Status': 'no_overlap'
                })
        except Exception as e:
            failed += 1
            results.append({
                'PointID': pid, 'Quarter': q,
                'Status': str(e)[:80]
            })

    # Progress
    if (done + failed) % 200 == 0:
        print(f"  {done + failed}/{total} tiles processed "
              f"({done} ok, {failed} failed)")

print(f"\nTiling complete.")
print(f"  Success: {done}")
print(f"  Failed:  {failed}")

if results:
    df_fail = pd.DataFrame(results)
    df_fail.to_csv(f'{TILE_OUTPUT}/failed_tiles.csv', index=False)
    print(f"  Failure log: {TILE_OUTPUT}/failed_tiles.csv")

In [ ]:
# ============================================================
# 6. VERIFY OUTPUT
# ============================================================

# Count tiles per point
tile_counts = []
for pid_str in os.listdir(TILE_OUTPUT):
    d = os.path.join(TILE_OUTPUT, pid_str)
    if os.path.isdir(d):
        n = len([f for f in os.listdir(d) if f.endswith('.tif')])
        tile_counts.append({'PointID': pid_str, 'n_tiles': n})

df_counts = pd.DataFrame(tile_counts)
print(f"Points with 4 tiles: {(df_counts['n_tiles'] == 4).sum()}")
print(f"Points with < 4 tiles: {(df_counts['n_tiles'] < 4).sum()}")
print(f"Points with 0 tiles: {(df_counts['n_tiles'] == 0).sum()}")

# Quick visual check on a random tile
import matplotlib.pyplot as plt

sample_dir = os.path.join(TILE_OUTPUT, df_counts.iloc[0]['PointID'])
sample_tifs = sorted([f for f in os.listdir(sample_dir) if f.endswith('.tif')])

if sample_tifs:
    fig, axes = plt.subplots(1, min(4, len(sample_tifs)), figsize=(16, 4))
    if not isinstance(axes, np.ndarray):
        axes = [axes]
    for ax, tif in zip(axes, sample_tifs[:4]):
        with rasterio.open(os.path.join(sample_dir, tif)) as src:
            img = src.read([1, 2, 3])
            img = np.transpose(img, (1, 2, 0)).astype(float)
            p2, p98 = np.percentile(img, (2, 98))
            if p98 > p2:
                img = np.clip((img - p2) / (p98 - p2), 0, 1)
            ax.imshow(img)
            ax.set_title(tif.split('_')[-1].replace('.tif', ''), fontsize=10)
            ax.axis('off')
    plt.suptitle(f'Sample tiles: Point {df_counts.iloc[0]["PointID"]}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7. SUMMARY
# ============================================================

total_tiles = df_counts['n_tiles'].sum()
total_points = len(df_counts)
complete = (df_counts['n_tiles'] == 4).sum()

print(f"{'='*50}")
print(f"TILING COMPLETE")
print(f"{'='*50}")
print(f"Total points:    {total_points}")
print(f"Complete (4/4):  {complete}")
print(f"Incomplete:      {total_points - complete}")
print(f"Total tiles:     {total_tiles}")
print(f"Output:          {TILE_OUTPUT}")
print(f"\nNext steps:")
print(f"  1. Extract VIIRS NTL median per point (Stage 2A)")
print(f"  2. Extract OSM features per point (Stage 2B)")
print(f"  3. Run VGG16 feature extraction on tiles (Stage 2C)")